# SNCP--PPO — Final Sistem: v34 Beta Politikası + v38 Eğitimsiz Eylem Kalkanı

Bu defter, çalışmanın **final sistemini** uçtan uca, sıfırdan üretir:

1. **Kurulum** — repo + bağımlılıklar (Colab GPU).
2. **v34 eğitimi** — v30 tabanı (ön-MLP + mean+max + yoğunluk müfredatı) **+ Beta eylem dağılımı**, 2.5M adım.
3. **Dürüst değerlendirme** — 5 tohum × 50 bölüm = 250 bölüm/yoğunluk (`paper_challenging`, robot 1.0 m/s).
4. **v38 eylem kalkanı** — eğitimsiz, yürütme-zamanı çarpışma filtresi; aynı v34 politikası üzerinde.
5. **İstatistik** — Wilson %95 GA, iki-oranlı z, Cohen h, Bonferroni; kalkanın etkisi **izole** edilir
   (v38 vs v34: aynı checkpoint / tohum / bölüm, tek fark shield).
6. **Görseller + yorum**.

> Önceki ablasyonların (v31–v37) hepsi yüksek yoğunluk açığını güvenilir biçimde kapatamadı; final
> çözüm yeni bir eğitim değil, mevcut v34 politikasının çarpışma-sınırı eylemini düzelten kısa-horizon
> bir güvenlik katmanıdır.

## 0. Çalışma zamanı (GPU)

In [ ]:
!nvidia-smi -L || echo "GPU yok — Çalışma Zamanı > Çalışma zamanı türünü değiştir > T4/A100 GPU seçin."

## 1. Kurulum: repo + bağımlılıklar

In [ ]:
import os
REPO = "sncp-ppo-crowdnav"
if not os.path.isdir(REPO):
    !git clone https://github.com/heimdilon/sncp-ppo-crowdnav.git
%cd {REPO}
!git pull --ff-only
!pip -q install -r requirements.txt
print("Kurulum tamam. CWD:", os.getcwd())

## 2. Final politikayı (v34 Beta) sıfırdan eğit

**Reçete (tek-değişken + Beta):** v30 tabanı = ön-MLP (Denklem 11) + mean+max dikkat havuzlama +
yoğunluk müfredatı $N\sim U(10,20)$; üzerine **Beta eylem dağılımı** (`--action_dist beta`) ve Beta'nın
entropi ölçeğine ayarlı entropi katsayısı (`--ent_coef 0.001`). Görünmez-robot/ORCA rejimi, robot 1.0 m/s,
2.5M adım, holdout-best checkpoint.

> A100'de ~3–4 saat. Eğitim sonunda en iyi holdout checkpoint'i `checkpoints/sncp_ppo_v34.pt`'ye yazılır.

In [ ]:
!python -m sncp_ppo.train \
  --num_envs 16 --horizon 128 --total_steps 2500000 --lr 1e-4 \
  --fixed_scenario paper_challenging --num_humans 10 --num_humans_range 10 20 \
  --bootstrap_easy_steps 200000 --robot_vpref 1.0 \
  --holdout_scenarios paper_standard paper_challenging --holdout_episodes 50 \
  --pre_mlp --meanmax_pool --action_dist beta --ent_coef 0.001 \
  --save_path checkpoints/sncp_ppo_v34.pt

In [ ]:
# Eğitim sonrası checkpoint'i doğrula (Beta başlığı otomatik algılanır)
import torch, os
assert os.path.exists("checkpoints/sncp_ppo_v34.pt"), "checkpoint bulunamadı — eğitim hücresini çalıştırın"
sd = torch.load("checkpoints/sncp_ppo_v34.pt", map_location="cpu")
print("checkpoint anahtar sayısı:", len(sd))
print("Beta politikası (actor_logstd YOK):", not any("actor_logstd" in k for k in sd))

## 3. Dürüst değerlendirme protokolü + v38 eylem kalkanı

Aşağıdaki yardımcı, **tek bir** `evaluate_density` yolu üzerinden hem ham politikayı (kalkan kapalı, **v34**)
hem de kalkanlı sistemi (kalkan açık, **v38**) aynı muhasebeyle değerlendirir:

* 5 tohum `[100,200,300,400,500]` × 50 bölüm = **250 bölüm/yoğunluk**, $N=5,10,15,20$.
* `paper_challenging`, robot 1.0 m/s, ORCA hız paritesi 1.0, görünmez robot, env-türevli süre bütçesi.
* **v38 kalkanı** = yürütmeden önce 6 adım (1.5 s) sabit-hız öngörüsü; çarpışma riski varsa küçük bir aday
  eylem kümesinden en güvenlisi seçilir (`action_shield=True`). PPO/checkpoint'e dokunmaz.

> Bu, v34 ve v38 için **aynı checkpoint, aynı tohumlar ve aynı bölümleri** kullanır; böylece v38−v34 farkı
> doğrudan kalkana atfedilebilir (model/tohum karıştırıcısı yok). Tam tarama ~30–45 dk sürer.

In [ ]:
import json, math, time
from sncp_ppo.eval_report import evaluate_density

CKPT = "checkpoints/sncp_ppo_v34.pt"
SEEDS = [100, 200, 300, 400, 500]
DENSITIES = [5, 10, 15, 20]
N_EP = 50

def pooled_se(p, n):
    return math.sqrt(p * (1 - p) / n) if n else float("nan")

def honest_sweep(action_shield, out_path, label):
    results = {}
    t0 = time.time()
    for N in DENSITIES:
        blocks, succ, coll, to, steps, isp = [], [], [], [], [], []
        for s in SEEDS:
            eps = evaluate_density(
                checkpoint_path=CKPT, num_humans=N, scenario="paper_challenging",
                n_episodes=N_EP, seed=s, robot_vpref=1.0, human_vpref_override=1.0,
                max_time=None, human_goal_noise=0.0,
                action_shield=action_shield, shield_horizon_steps=6, shield_safety_margin=0.0,
            )
            b = [e.success for e in eps]
            blocks.append(sum(b) / len(b))
            succ += b
            coll += [e.collision for e in eps]
            to += [e.timeout for e in eps]
            steps += [e.steps for e in eps if e.success]
            isp += [e.avg_i_sp for e in eps]
            print(f"  [{label}] N={N} seed={s} succ={blocks[-1]*100:4.1f}%  ({time.time()-t0:.0f}s)", flush=True)
        n = len(succ); p = sum(succ) / n
        results[str(N)] = {
            "block_means": blocks, "pooled_success": p, "pooled_se": pooled_se(p, n),
            "pooled_collision": sum(coll) / n, "pooled_timeout": sum(to) / n,
            "avg_success_steps": (sum(steps) / len(steps)) if steps else float("nan"),
            "avg_i_sp": sum(isp) / len(isp), "n": n,
        }
        r = results[str(N)]
        print(f"== [{label}] N={N} POOLED succ={p*100:.1f}%  coll={r['pooled_collision']*100:.1f}%  to={r['pooled_timeout']*100:.1f}% ==", flush=True)
        json.dump(results, open(out_path, "w"), indent=2)
    return results

v34 = honest_sweep(action_shield=False, out_path="v34_multiseed_result.json", label="v34 ham")
v38 = honest_sweep(action_shield=True,  out_path="v38_multiseed_result.json", label="v38 kalkan")
print("\nDÜRÜST TARAMA TAMAM -> v34_multiseed_result.json, v38_multiseed_result.json")

## 4. İstatistiksel analiz — v38 (kalkan) vs v34 (ham)

Kalkanın etkisi izole edilir. Bonferroni eşiği $\alpha = 0.05/4 = 0.0125$ (4 yoğunluk; başarı ve çarpışma
ayrı önceden-kayıtlı test aileleri).

In [ ]:
import json, math
from scipy.stats import norm

Z = 1.959963984540054
DENS = [5, 10, 15, 20]
ALPHA = 0.0125
v34 = json.load(open("v34_multiseed_result.json"))
v38 = json.load(open("v38_multiseed_result.json"))

def wilson(k, n):
    p = k / n; d = 1 + Z*Z/n; c = (p + Z*Z/(2*n)) / d
    h = Z*math.sqrt(p*(1-p)/n + Z*Z/(4*n*n)) / d
    return (c-h)*100, (c+h)*100

def ztest(k1, n1, k2, n2):
    p1, p2 = k1/n1, k2/n2; pp = (k1+k2)/(n1+n2)
    se = math.sqrt(pp*(1-pp)*(1/n1+1/n2)); z = (p1-p2)/se if se > 0 else 0.0
    return z, 2*norm.sf(abs(z))

def cohen_h(p1, p2):
    return 2*(math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))

print(f"{'N':>3} | {'v34 succ':>8} | {'v38 succ [95% GA]':>22} | {'dPP succ (p)':>16} | {'v34 coll':>8} | {'v38 coll':>8} | {'dPP coll (p)':>16}")
print("-" * 104)
for N in DENS:
    s = str(N); n = v38[s]["n"]
    ks, kb = round(v38[s]["pooled_success"]*n), round(v34[s]["pooled_success"]*n)
    lo, hi = wilson(ks, n); zs, ps = ztest(ks, n, kb, n)
    cc, cb = round(v38[s]["pooled_collision"]*n), round(v34[s]["pooled_collision"]*n)
    zc, pc = ztest(cc, n, cb, n)
    ds = (v38[s]["pooled_success"]-v34[s]["pooled_success"])*100
    dc = (v38[s]["pooled_collision"]-v34[s]["pooled_collision"])*100
    gs = "*" if ps < ALPHA else " "; gc = "*" if pc < ALPHA else " "
    print(f"{N:>3} | {v34[s]['pooled_success']*100:>7.1f} | {v38[s]['pooled_success']*100:>6.1f} [{lo:>5.1f},{hi:>5.1f}] | {ds:>+6.1f} ({ps:6.4f}){gs} | {v34[s]['pooled_collision']*100:>7.1f} | {v38[s]['pooled_collision']*100:>7.1f} | {dc:>+6.1f} ({pc:6.4f}){gc}")

print("\n*  = Bonferroni-anlamlı (p < 0.0125).")
print("Cohen h (N=20 başarı):", round(cohen_h(v38['20']['pooled_success'], v34['20']['pooled_success']), 3))

## 5. Görseller: karşılaştırma figürü + kalkanlı yörüngeler

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
v34 = json.load(open("v34_multiseed_result.json"))
v38 = json.load(open("v38_multiseed_result.json"))
N = np.array([5, 10, 15, 20])
s34 = np.array([v34[str(n)]["pooled_success"]*100 for n in N]);   c34 = np.array([v34[str(n)]["pooled_collision"]*100 for n in N])
s38 = np.array([v38[str(n)]["pooled_success"]*100 for n in N]);   c38 = np.array([v38[str(n)]["pooled_collision"]*100 for n in N])
def se(a):
    p = a/100.0; return 1.96*np.sqrt(p*(1-p)/250.0)*100.0

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
a1.plot(N, s34, "--s", color="#BA7517", lw=2, ms=7, label="v34 (ham Beta)")
a1.errorbar(N, s38, yerr=se(s38), fmt="-D", color="#3B6D11", lw=2.6, ms=8, capsize=3, label="v38 (+ eylem kalkanı)")
a1.axhline(94, color="gray", ls=":", label="Makale ~%94")
a1.set_title("Başarı"); a1.set_xlabel("Yaya sayısı N"); a1.set_ylabel("Başarı (%)"); a1.set_xticks(N); a1.set_ylim(80, 101); a1.legend(); a1.grid(alpha=.3)
a2.plot(N, c34, "--s", color="#BA7517", lw=2, ms=7, label="v34")
a2.errorbar(N, c38, yerr=se(c38), fmt="-D", color="#3B6D11", lw=2.6, ms=8, capsize=3, label="v38 (kalkan)")
a2.set_title("Çarpışma (kalkan ≈%0'a indirir)"); a2.set_xlabel("Yaya sayısı N"); a2.set_ylabel("Çarpışma (%)"); a2.set_xticks(N); a2.set_ylim(-1, 25); a2.legend(); a2.grid(alpha=.3)
fig.tight_layout(); fig.savefig("v38_final_comparison.png", dpi=150); plt.show()

In [ ]:
# Kalkanlı yörüngeler (N=10 ve N=20) — kalkan sosyal rotayı korur, çarpışma-sınırı eylemini düzeltir
from sncp_ppo.eval_report import render_trajectory
from IPython.display import Image, display
for n in (10, 20):
    render_trajectory(
        checkpoint_path="checkpoints/sncp_ppo_v34.pt", output_path=f"v38_traj_n{n}.png",
        num_humans=n, scenario="paper_challenging", seed=100,
        robot_vpref=1.0, human_vpref_override=1.0, max_time=50.0, action_shield=True,
    )
    display(Image(f"v38_traj_n{n}.png"))

## 6. Sonuç ve yorum

* **v34 (ham Beta politikası)** güçlü bir sosyal rota üreticisidir; ancak yüksek yoğunlukta kalan
  başarısızlıkların çoğu **çarpışma** kaynaklıdır (zaman aşımı değil).
* **v38 eylem kalkanı** — yeni bir eğitim/checkpoint olmadan — bu çarpışma-sınırı eylemini düzeltir:
  dürüst 5-tohum havuzda çarpışma her yoğunlukta pratikte **≈%0'a** iner, başarı **≥%98.8** olur ve
  v34'e göre N=10/15/20 kazancı **Bonferroni-anlamlıdır** (yukarıdaki tablo).
* Kalkan, öğrenilmiş politikanın yerine geçmez; onu daha güvenli bir **yürütme sistemine** dönüştürür.
  Bu, pratik robotik için doğrudan ve düşük-riskli bir mühendislik çözümüdür.

**Üretilen artefaktlar:** `checkpoints/sncp_ppo_v34.pt`, `v34_multiseed_result.json`,
`v38_multiseed_result.json`, `v38_final_comparison.png`, `v38_traj_n10.png`, `v38_traj_n20.png`.

> Kalkanın varsayımı: yayaların kısa-horizon **sabit-hız** hareketi ve değerlendirmede ortamın gerçek
> yaya konum/hızları. Gerçek bir robotta bu bilgiler algılama gürültüsüyle gelir.

## 7. Artefaktları indir (Colab)

Final çıktıları (sweep JSON'ları, karşılaştırma figürü, yörüngeler, checkpoint) tek ZIP'te toplar ve
indirir — makaleye / teslime taşımak için.

In [ ]:
import os, zipfile
DOWNLOAD = True
bundle = "sncp_ppo_v38_final_artifacts.zip"
artifacts = [
    "v34_multiseed_result.json", "v38_multiseed_result.json",
    "v38_final_comparison.png", "v38_traj_n10.png", "v38_traj_n20.png",
    "checkpoints/sncp_ppo_v34.pt",
]
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in artifacts:
        if os.path.exists(f):
            z.write(f); print("  +", f)
print("paket:", bundle)
if DOWNLOAD:
    try:
        from google.colab import files
        files.download(bundle)
    except Exception as e:
        print("indirme atlandı (Colab dışı ortam):", e)